In [5]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone



In [6]:

# ==============================
# 1) OAuth2 credentials
# ==============================
CLIENT_ID = "amir711-api-client"     
CLIENT_SECRET = "jY5i0NH2d42DYdYakPwPuGltyMO7EuWJ"   

TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"

def get_access_token():
    """
    Get OAuth2 access token from OpenSky using client credentials.
    """
    data = {
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    }

    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }

    response = requests.post(TOKEN_URL, data=data, headers=headers)
    print("Token status:", response.status_code)

    if response.status_code != 200:
        print("Token error:", response.text)
        raise RuntimeError("Failed to get access token")

    token_info = response.json()
    return token_info["access_token"]


def get_flights_all(access_token, begin_dt, end_dt):
    """
    Call /flights/all with Bearer token between begin_dt and end_dt (UTC datetimes).
    NOTE: OpenSky بيحسب الـ flights من البيانات التاريخية،
    فغالبًا لازم تستخدم أيام سابقة مش اليوم الحالي. :contentReference[oaicite:1]{index=1}
    """
    begin = int(begin_dt.timestamp())
    end = int(end_dt.timestamp())

    url = "https://opensky-network.org/api/flights/all"
    headers = {
        "Authorization": f"Bearer {access_token}"
    }
    params = {
        "begin": begin,
        "end": end
    }

    print("Requesting flights/all ...")
    print("URL:", url)
    print("Params:", params)

    r = requests.get(url, headers=headers, params=params)
    print("Flights status:", r.status_code)

    if r.status_code != 200:
        print("Flights error:", r.text)
        raise RuntimeError("Failed to get flights")

    flights = r.json()
    df = pd.DataFrame(flights)
    return df



In [7]:

if __name__ == "__main__":

    today_utc = datetime.now(timezone.utc).date()
    yesterday_utc = today_utc - timedelta(days=1)

    begin_dt = datetime(yesterday_utc.year, yesterday_utc.month, yesterday_utc.day, 0, 0, 0, tzinfo=timezone.utc)
    end_dt = begin_dt + timedelta(days=1)

    print("Begin (UTC):", begin_dt)
    print("End   (UTC):", end_dt)


    token = get_access_token()
    print("Got access token.")


    df_flights = get_flights_all(token, begin_dt, end_dt)
    print("Number of flights:", len(df_flights))
    print(df_flights.head())



    out_file = "opensky_flights_yesterday.csv"
    df_flights.to_csv(out_file, index=False, encoding="utf-8")
    print(f"Saved flights to {out_file}")

Begin (UTC): 2025-12-11 00:00:00+00:00
End   (UTC): 2025-12-12 00:00:00+00:00
Token status: 200
Got access token.
Requesting flights/all ...
URL: https://opensky-network.org/api/flights/all
Params: {'begin': 1765411200, 'end': 1765497600}
Flights status: 200
Number of flights: 117935
   icao24   firstSeen estDepartureAirport    lastSeen estArrivalAirport  \
0  a54beb  1765411204                None  1765414370              KAGC   
1  a57d42  1765473370                KDFW  1765489985              KEWR   
2  a57d42  1765460046                KPHX  1765467171              KDFW   
3  a58bab  1765486330                KSFB  1765490692              KOCF   
4  a58ded  1765457014                KALB  1765463248              07KY   

   callsign  estDepartureAirportHorizDistance  \
0  PJC40                                  NaN   
1  AAL2852                              756.0   
2  AAL2103                              861.0   
3  N45662                              1964.0   
4  UPS2121         

In [8]:

flights_file = "opensky_flights_yesterday.csv"
df = pd.read_csv(flights_file)

print("Columns in flights df:")
print(df.columns)

required_cols = ["icao24", "firstSeen", "lastSeen", "estDepartureAirport", "estArrivalAirport", "callsign"]
for c in required_cols:
    if c not in df.columns:
        print(f"WARNING: column {c} not found in dataframe!")

df["firstSeen_dt"] = pd.to_datetime(df["firstSeen"], unit="s", utc=True)
df["lastSeen_dt"] = pd.to_datetime(df["lastSeen"], unit="s", utc=True)


df["case:concept:name"] = df["icao24"].astype(str) + "_" + df["firstSeen"].astype(str)

dep_events = pd.DataFrame({
    "case:concept:name": df["case:concept:name"],
    "concept:name": "DEPARTURE",
    "time:timestamp": df["firstSeen_dt"],
    "icao24": df["icao24"],
    "callsign": df.get("callsign"),
    "origin": df.get("estDepartureAirport"),
    "destination": df.get("estArrivalAirport"),
})

arr_events = pd.DataFrame({
    "case:concept:name": df["case:concept:name"],
    "concept:name": "ARRIVAL",
    "time:timestamp": df["lastSeen_dt"],
    "icao24": df["icao24"],
    "callsign": df.get("callsign"),
    "origin": df.get("estDepartureAirport"),
    "destination": df.get("estArrivalAirport"),
})

event_log = pd.concat([dep_events, arr_events], ignore_index=True)

event_log = event_log.dropna(subset=["time:timestamp"])


event_log = event_log.sort_values(by=["case:concept:name", "time:timestamp"])

print("Sample of event_log:")
print(event_log.head(10))

out_file = "turnaround_log_opensky.csv"
event_log.to_csv(out_file, index=False)
print(f"Saved event log to {out_file}")


Columns in flights df:
Index(['icao24', 'firstSeen', 'estDepartureAirport', 'lastSeen',
       'estArrivalAirport', 'callsign', 'estDepartureAirportHorizDistance',
       'estDepartureAirportVertDistance', 'estArrivalAirportHorizDistance',
       'estArrivalAirportVertDistance', 'departureAirportCandidatesCount',
       'arrivalAirportCandidatesCount'],
      dtype='object')
Sample of event_log:
        case:concept:name concept:name            time:timestamp  icao24  \
111662  000001_1765411204    DEPARTURE 2025-12-11 00:00:04+00:00  000001   
229597  000001_1765411204      ARRIVAL 2025-12-11 08:08:27+00:00  000001   
111661  000001_1765441620    DEPARTURE 2025-12-11 08:27:00+00:00  000001   
229596  000001_1765441620      ARRIVAL 2025-12-11 08:41:28+00:00  000001   
111660  000001_1765446041    DEPARTURE 2025-12-11 09:40:41+00:00  000001   
229595  000001_1765446041      ARRIVAL 2025-12-11 09:51:50+00:00  000001   
111659  000001_1765453325    DEPARTURE 2025-12-11 11:42:05+00:00  000